# 电子邮件

## SMTP发送邮件

In [ ]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders

# ========== 配置区（改成自己信息）==========
smtp_server = "smtp.qq.com"    # QQ发件服务器
smtp_port = 465                # SSL端口
sender = "xxx@qq.com"          # 发件邮箱
auth_code = "xxxxxxx"          # QQ邮箱授权码（邮箱后台开启IMAP生成）
receiver = "目标邮箱@xxx.com"   # 收件邮箱
mail_title = "Python测试邮件"   # 邮件标题
file_path = "test.txt"         # 附件路径，不需要附件可删除附件代码
# ==========================================

# 1. 创建多类型邮件对象（可同时：正文+附件）
msg = MIMEMultipart()
# 邮件头部信息
msg["From"] = f"Python脚本 <{sender}>"  # 发件人显示名称+邮箱
msg["To"] = receiver                   # 收件人
msg["Subject"] = mail_title            # 标题

# 2. 添加邮件正文（plain纯文本，换成html可写网页格式）
content = "你好，这是Python email模块自动发送的测试邮件！\n附件为测试文件"
text_part = MIMEText(content, "plain", "utf-8")
msg.attach(text_part)

# 3.【可选】添加附件，不需要附件直接删掉本段
try:
    with open(file_path, "rb") as f:
        base = MIMEBase("application", "octet-stream")
        base.set_payload(f.read())
    # base64编码附件
    encoders.encode_base64(base)
    # 设置附件文件名
    base.add_header("Content-Disposition", "attachment", filename=file_path)
    msg.attach(base)
except Exception as e:
    print("附件读取失败，跳过附件：", e)

# 4. SSL连接SMTP服务器，发送邮件
try:
    # 465端口用SSL加密
    server = smtplib.SMTP_SSL(smtp_server, smtp_port)
    # 登录邮箱：账号+授权码
    server.login(sender, auth_code)
    # 发送：发件人、收件人列表、邮件字符串
    server.sendmail(sender, receiver, msg.as_string())
    server.quit()
    print("✅ 邮件发送成功！")
except Exception as err:
    print("❌ 发送失败：", err)

## POP3收取邮件

In [ ]:
import poplib
from email.parser import Parser
from email.header import decode_header
from email.utils import parseaddr

# =====================配置区=====================
POP3_SERVER = "pop.qq.com"
POP3_SSL_PORT = 995       # SSL加密端口
EMAIL_ACCOUNT = "你的QQ@qq.com"
EMAIL_AUTH_CODE = "邮箱授权码"  # 邮箱POP3授权码
# ================================================

def decode_str(s):
    """解码邮件标题、发件人中文"""
    value, charset = decode_header(s)[0]
    if charset:
        value = value.decode(charset)
    return value

def get_email_info(msg):
    """解析单封邮件：发件人、标题、正文"""
    # 解析发件人
    from_addr = parseaddr(msg.get('From'))[1]
    from_name = decode_str(parseaddr(msg.get('From'))[0])
    # 解析标题
    title = decode_str(msg.get("Subject"))

    # 提取正文
    content = ""
    if msg.is_multipart():
        # 多部分(正文+附件)循环遍历
        for part in msg.get_payload():
            # 只取文本正文
            if part.get_content_type() in ["text/plain", "text/html"]:
                payload = part.get_payload(decode=True)
                charset = part.get_charset()
                if charset is None:
                    charset = part.get("Content-Type", "").split("charset=")[-1].strip()
                content = payload.decode(charset)
                break
    else:
        # 纯文本邮件
        payload = msg.get_payload(decode=True)
        charset = msg.get_charset()
        content = payload.decode(charset)

    info = {
        "from_name": from_name,
        "from_addr": from_addr,
        "title": title,
        "content": content.strip()
    }
    return info

def pop_receive_mail():
    # 1. 连接POP3 SSL服务
    pop_conn = poplib.POP3_SSL(POP3_SERVER, POP3_SSL_PORT)
    # 2. 登录账号+授权码
    pop_conn.user(EMAIL_ACCOUNT)
    pop_conn.pass_(EMAIL_AUTH_CODE)

    # stat(): (邮件总数, 总字节大小)
    total_mail, total_bytes = pop_conn.stat()
    print(f"📩 收件箱总邮件数：{total_mail}")
    if total_mail == 0:
        pop_conn.quit()
        return

    # list() 获取所有邮件编号与大小：[(b'1 2345'), (b'2 5678')...]
    resp, mail_id_list, octets = pop_conn.list()
    mail_ids = [int(i.split()[0]) for i in mail_id_list]

    # 取最新1封邮件(最后一个编号)
    latest_id = mail_ids[-1]
    print(f"\n正在读取第 {latest_id} 号邮件")

    # retr(编号): 获取整封邮件原始字节数据
    resp, mail_lines, oct = pop_conn.retr(latest_id)
    mail_raw = b"\r\n".join(mail_lines).decode("utf-8", errors="ignore")

    # 原始字符串转为邮件Message对象
    msg = Parser().parsestr(mail_raw)
    mail_data = get_email_info(msg)

    # 打印结果
    print("="*50)
    print(f"发件人昵称：{mail_data['from_name']}")
    print(f"发件邮箱：{mail_data['from_addr']}")
    print(f"邮件标题：{mail_data['title']}")
    print(f"邮件正文：\n{mail_data['content']}")

    # 关闭连接
    pop_conn.quit()

if __name__ == "__main__":
    pop_receive_mail()